# Ядро «C (интерактивный)»: справка

Ячейка с `main` это обычная программа на Си: ядро компилирует её (`gcc -O2 -Wall -lm`) и запускает. Ввод и настройки пишутся **в комментариях** в конце ячейки, поэтому для компилятора это обычный код.

| Что написать в ячейке | Что делает |
|---|---|
| ничего | ввод с клавиатуры на каждом `scanf`, слово `EOF` заканчивает ввод |
| `/*%input ... */` | текстовый ввод; несколько блоков = несколько тестов |
| `//%file: input.txt` | ввод из файла |
| `/*%bytes ... */`, `//%ints: 1 -1 10` | двоичный ввод: байты (hex) или числа по 4 байта |
| `//%out: hex`, `//%out: ints`, `//%out: bytes` | показать двоичный вывод `fwrite` |
| `//%time` | время работы |
| `//%cflags: -O0`, `//%args: a b` | флаги gcc, аргументы `argv` |
| `/*%gdb ... */` | запустить под отладчиком gdb с этими командами |
| `//%debug: 21 35` | подготовить отладку в интерфейсе VS Code: F5, остановка на строках 21 и 35 |

Ячейки **без `main`**, как в блокноте Python, работают вместе с программами:

| Ячейка без `main` | Что делает |
|---|---|
| только функции, структуры, `#include` | запоминаются и добавляются ко всем следующим программам |

Как в Python, функции из уже запущенных ячеек видны дальше: если программа вызывает функцию, которой в ней нет, ядро возьмёт её из ранее запущенной ячейки (всё, кроме её `main`).
| выражения без `;`: `N, A[0:N]`, `Items[0:N].Key` | показать значения в последней программе, без `printf`; у среза массива структур можно взять одно поле |
| код с `;` | выполнить его внутри последней программы |
| только блок `/*%gdb` | отладить последнюю программу |
| только `//%debug: 21` | подготовить последнюю программу к отладке в VS Code (F5) |
| `//%line: 10`, `//%test: 2` | где остановиться (перед строкой 10) и на каком тесте; несколько `//%line` в одной ячейке смотрят в разные места |

Ещё: кнопка Stop останавливает программу; ячейка с ошибкой становится красной; если программа упала, видно строку и переменные.

## Ввод

Без блока ввода на каждом `scanf` появится поле ввода (в VS Code сверху).

In [1]:
#include <stdio.h>

int main(void)
{
    int A, B;

    scanf("%d %d", &A, &B);
    printf("%d\n", A + B);
    return 0;
}

2 3
5


Каждый блок `/*%input` это отдельный запуск, так удобно проверять примеры из условия.

In [2]:
#include <stdio.h>

int main(void)
{
    int A, B;

    scanf("%d %d", &A, &B);
    printf("%d\n", A + B);
    return 0;
}

/*%input
2 3
*/

/*%input
10 -4
*/

--- тест 1 ---
5

--- тест 2 ---
6


Из файла (путь от папки блокнота). После конца ввода `scanf` возвращает не 1, и цикл заканчивается.

In [3]:
#include <stdio.h>

int main(void)
{
    int Number, Sum = 0;

    while (scanf("%d", &Number) == 1)
        Sum += Number;
    printf("%d\n", Sum);
    return 0;
}

//%file: input.txt

15


## Двоичный ввод и вывод

Байты можно скопировать прямо из условия. `//%out` показывает то, что записал `fwrite`.

In [4]:
#include <stdio.h>

int main(void)
{
    int Number;

    while (fread(&Number, sizeof(int), 1, stdin) == 1)
    {
        Number *= 2;
        fwrite(&Number, sizeof(int), 1, stdout);
    }
    return 0;
}

/*%bytes
01 00 00 00 FF FF FF FF 0A 00 00 00
*/
//%out: hex

02 00 00 00 FE FF FF FF 14 00 00 00


In [5]:
#include <stdio.h>

int main(void)
{
    int Number;

    while (fread(&Number, sizeof(int), 1, stdin) == 1)
    {
        Number *= 2;
        fwrite(&Number, sizeof(int), 1, stdout);
    }
    return 0;
}

//%ints: 1 -1 10
//%out: ints

2 -2 20


## Определения в отдельной ячейке

Как `def` в Python: функция запоминается и доступна в следующих программах. Забыть всё: перезапустить ядро (Restart).

In [6]:
#include <stdio.h>

int Square(int Value)
{
    return Value * Value;
}

[определения запомнены, они будут во всех следующих программах]


In [7]:
int main(void)
{
    printf("%d\n", Square(7));
    return 0;
}

49


Функции из ячейки с `main` тоже можно вызывать в следующих ячейках, например чтобы проверить их на маленьком примере. Ядро возьмёт их из ранее запущенной ячейки и напишет, что взяло.

In [8]:
#include <stdio.h>

int Max(int First, int Second)
{
    return First > Second ? First : Second;
}

int main(void)
{
    printf("%d\n", Max(3, 7));
    return 0;
}

7


In [9]:
int main(void)
{
    printf("%d\n", Max(-1, -5));
    return 0;
}

[из ранее запущенной ячейки взято: Max]
-1


## Значения переменных без printf

Напишите в ячейке без `main` выражения через запятую. Значения берутся в конце `main` последней запущенной программы.

In [10]:
#include <stdio.h>

typedef struct
{
    int X, Y;
} Point;

int main(void)
{
    int Numbers[4] = {4, 2, 3, 1};
    Point P = {3, 4};
    char Name[] = "точка";

    Numbers[0] = P.X + P.Y;
    printf("%s %d\n", Name, Numbers[0]);
    return 0;
}

точка 7


In [11]:
Numbers, P, Name, P.X * P.Y

[строка 16 программы, тест 1]
Numbers = {7, 2, 3, 1}
P = {
  X = 3,
  Y = 4
}
Name = "точка"
P.X * P.Y = 12


Массив из `malloc` это только адрес, его элементы показывает **срез как в Python**: `A[0:N]`, `A[:N]`, `A[1:3]`.

In [12]:
#include <stdlib.h>

int main(void)
{
    int N = 5;
    int *Squares = malloc(N * sizeof(int));

    for (int i = 0; i < N; ++i)
        Squares[i] = i * i;
    free(Squares);
    return 0;
}

In [13]:
Squares, Squares[0:N], Squares[1:3]

[строка 10 программы, тест 1]
Squares = 0x5555555592a0    (это адрес; элементы покажет срез, например Squares[0:N])
Squares[0:N] = {0, 1, 4, 9, 16}
Squares[1:3] = {1, 4}


`//%line: 9` показывает значения перед строкой 9 (номера слева). Строка в цикле выполняется много раз, поэтому значения выводятся по проходам.

In [14]:
//%line: 9
i

[строка 9 программы, тест 1]
--- проход 1 ---
i = 0
--- проход 2 ---
i = 1
--- проход 3 ---
i = 2
--- проход 4 ---
i = 3
--- проход 5 ---
i = 4


Строки с `;` выполняются внутри программы, перед строкой `//%line` (без неё в конце `main`).

In [15]:
for (int i = 0; i < N; ++i)
    printf("%d ", Squares[i]);

[строка 10 программы, тест 1]
0 1 4 9 16 


## Отладчик gdb

Ячейка с одним блоком `/*%gdb` отлаживает последнюю программу.

In [16]:
/*%gdb
break 9
run
print i
continue
print i
*/

Breakpoint 1 at 0x11ad: file ./cell.c, line 9.

Breakpoint 1, main () at ./cell.c:9
9	        Squares[i] = i * i;
$1 = 0

Breakpoint 1, main () at ./cell.c:9
9	        Squares[i] = i * i;
$2 = 1


## Несколько мест в одной ячейке и поля структур

Каждый `//%line` начинает новую часть ячейки: выражения и код под ним работают перед своей строкой. У среза массива структур можно взять одно поле: `Items[0:N].Key`.

In [17]:
#include <stdio.h>
#include <stdlib.h>

typedef struct
{
    int Key, Value;
} Item;

int Compare(const void *Left, const void *Right)
{
    return ((Item *)Left)->Key - ((Item *)Right)->Key;
}

int main(void)
{
    int N;
    scanf("%d", &N);
    Item *Items = malloc(N * sizeof(Item));
    for (int i = 0; i < N; ++i)
        scanf("%d %d", &Items[i].Key, &Items[i].Value);
    qsort(Items, N, sizeof(Item), Compare);
    for (int i = 0; i < N; ++i)
        printf("%d %d\n", Items[i].Key, Items[i].Value);
    free(Items);
    return 0;
}

/*%input
3
5 50
1 10
3 30
*/

1 10
3 30
5 50


In [18]:
//%line: 21
Items[0:N]
//%line: 24
Items[0:N].Key, Items[0:N].Value

[строка 21 программы, тест 1]
Items[0:N] = {{Key = 5, Value = 50}, {Key = 1, Value = 10}, {Key = 3, Value = 30}}
[строка 24 программы, тест 1]
Items[0:N].Key = {1, 3, 5}
Items[0:N].Value = {10, 30, 50}


## Отладка в интерфейсе VS Code

Кнопка «Debug Cell» в VS Code работает только в блокнотах Python, поэтому для Си отладка запускается так:

1. Ячейка `//%debug: 21 24` (или строка `//%debug: ...` в самой программе) собирает последнюю программу для отладчика с вводом из её примера (`//%test: 2` выбирает другой).
2. Нажмите **F5** (или слева Run and Debug -> **C: отладка ячейки** -> зелёный треугольник). Конфигурация сама добавляется в `.vscode/launch.json`.
3. Откроется код ячейки, программа остановится на строке 21. VS Code пишет «Paused on exception: Hit breakpoint», это и есть остановка.
4. Слева **Variables**: переменные текущей функции; **Watch**: любые выражения, например `*Items@N` (массив из `malloc` целиком), `Items[1].Key`; **Call Stack**: цепочка вызовов.
5. **F10** шаг, **F11** зайти в функцию, **Shift+F11** выйти из неё, **F5** до следующей остановки. Точки останова можно ставить и мышкой слева от номера строки в открывшемся файле.

Нужно расширение VS Code **C/C++** (ms-vscode.cpptools); без `//%debug` в строке программа остановится в начале `main`.

In [19]:
//%debug: 21 24

[отладка в VS Code готова: F5 -> «C: отладка ячейки», тест 1, остановка на строках 21, 24]


## Падение программы

Ядро само показывает цепочку вызовов и переменные в момент падения.

In [20]:
#include <stdio.h>

int Get(int *Numbers, int Index)
{
    return Numbers[Index];
}

int main(void)
{
    int Numbers[3] = {1, 2, 3};

    printf("%d\n", Get(Numbers, 1000000));
    return 0;
}

[C kernel] программа упала: SIGSEGV, обращение к чужой памяти
где упала (сначала самый внешний вызов):
  main, строка 12:  printf("%d\n", Get(Numbers, 1000000));
  Get, строка 5:  return Numbers[Index];
переменные в Get:
  Numbers = 0x7fffffffd7cc    (это адрес; элементы покажет срез, например Numbers[0:N])
  Index = 1000000


## Время и остановка

`//%time` печатает время работы. Зависшую программу останавливает кнопка Stop.

In [21]:
//%time
#include <stdio.h>

int main(void)
{
    long long Sum = 0;

    for (int i = 0; i < 300000000; ++i)
        Sum += i % 7;
    printf("%lld\n", Sum);
    return 0;
}

899999997
[время: 0.473 с]


In [22]:
int main(void)
{
    while (1);
}

[C kernel] программа остановлена
